# Quantized weights: a GGUF checkpoint, served and fine-tuned

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.55.0/cookbook/notebooks/book/precision/quantized-weights.ipynb)

Built from [`cookbook/book/chapters/precision/quantized-weights.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/precision/quantized-weights.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.55.0 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed samples with a compact text encoder: seconds
# to minutes, or for a chapter that fine-tunes an encoder, minutes on a GPU and up
# to hours on a CPU. The models it runs are downloaded from the Hugging Face Hub on
# first use. SCALE = "full" runs it over the published data and a larger text
# encoder instead, on the GPU: the chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.55.0", engine + "==0.55.0", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.55.0#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

**Verbs:** a `local:` model directory holding `model.gguf` and its `config.json` ·
`generate_embeddings` · `describe_table` · `fine_tune(..., target_modules=[...])`
over a quantized base (QLoRA) · `job.metrics()` · **Background:** block
quantization as a frozen-backbone compression, and LoRA trained over it
(Dettmers et al. 2023) · **Checked:** live (a real encoder quantized to `q8_0` here;
its vectors compared with the full-precision checkpoint's, both tables'
identities read back, and a QLoRA fine-tune's held-out loss measured).

## A fact about the file, not a setting

[Compute precision](https://f-inverse.github.io/jammi-ai/cookbook/chapters/precision/compute-precision.html) is a setting: the dtype a forward pass
runs at. Weight quantization is a fact about the checkpoint. A model directory that
holds a `model.gguf` instead of a `model.safetensors` has its weights stored
block-quantized, and Jammi loads it through the same `local:` path as any other
checkpoint: the quantized matrices stay compressed in memory, and everything else
(embeddings, layer norms, biases) is expanded to the compute dtype. No argument of
`generate_embeddings` names a quantization; the directory decides.

Jammi reads a GGUF file with Hugging Face tensor names beside the checkpoint's own
`config.json`. Here one is written from a real sentence encoder,
`sentence-transformers/all-MiniLM-L6-v2`: every matrix multiplied in a layer (the
attention projections and the feed-forward pair) is quantized to `q8_0` — one `f16`
scale per block of 32 weights, each weight an 8-bit integer — and the rest is kept
at `f32`.

In [ ]:
import json
import math
import shutil
import tempfile
from pathlib import Path

import jammi
from huggingface_hub import snapshot_download
from jammi_cookbook import fixtures
from jammi_cookbook.claims import claim
from jammi_cookbook.gguf_fixture import write_q8_0_gguf

BASE = "sentence-transformers/all-MiniLM-L6-v2"
base_dir = Path(snapshot_download(BASE, allow_patterns=["config.json", "model.safetensors",
                                                         "tokenizer.json"]))
num_layers = json.loads((base_dir / "config.json").read_text())["num_hidden_layers"]

work = Path(tempfile.mkdtemp(prefix="quantized_weights_"))
dense_dir, gguf_dir = work / "dense", work / "q8_0"
for d in (dense_dir, gguf_dir):
    d.mkdir()
    for name in ("config.json", "tokenizer.json"):
        shutil.copy(base_dir / name, d / name)
shutil.copy(base_dir / "model.safetensors", dense_dir / "model.safetensors")
write_q8_0_gguf(base_dir, num_layers=num_layers, dest_dir=gguf_dir)

dense_bytes = (dense_dir / "model.safetensors").stat().st_size
gguf_bytes = (gguf_dir / "model.gguf").stat().st_size
print(f"f32 safetensors: {dense_bytes / 1e6:.1f} MB    q8_0 gguf: {gguf_bytes / 1e6:.1f} MB")

## The quantized encoder points where the full one does

Both directories embed the same corpus through the same call; only the `local:`
path differs. Each quantized vector is compared with its full-precision twin by
cosine:

In [ ]:
db = jammi.connect(f"file://{work / 'engine'}")
db.add_source("corpus", url=fixtures.url("tiny_corpus.parquet"), format="parquet")


def embed(model_dir: Path) -> tuple[str, dict]:
    table = db.generate_embeddings(source="corpus", model=f"local:{model_dir}",
                                   columns=["content"], key="id")
    rows = db.sql(f'SELECT _row_id, vector FROM "jammi.{table}"').to_pylist()
    return table, {r["_row_id"]: r["vector"] for r in rows}


def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / math.sqrt(sum(x * x for x in a) * sum(y * y for y in b))


table_dense, dense = embed(dense_dir)
table_gguf, quantized = embed(gguf_dir)
agreement = min(cosine(dense[k], quantized[k]) for k in dense)
print(f"lowest cosine between a q8_0 vector and its f32 twin: {agreement:.6f}")

claim("the q8_0 checkpoint is smaller on disk than the f32 one", gguf_bytes < dense_bytes,
      f"{gguf_bytes / dense_bytes:.0%} of the f32 size")
claim("q8_0 weights point every vector close to where the f32 weights do", agreement > 0.995,
      f"lowest cosine {agreement:.6f}")

What `q8_0` saves is bounded by what stays dense. MiniLM's word-embedding table
(30,522 × 384) is not a matrix multiplied in a layer, so it stays at `f32`; a model
whose layers outweigh its vocabulary saves proportionally more. The cost — a cosine
a little under 1 — is far below what [the storage-precision chapters](https://f-inverse.github.io/jammi-ai/cookbook/chapters/precision/precision.html)'
`int8` and `binary` indexes give up, which costs recall that a rescoring stage has
to win back.

## A quantized table is another definition

A quantized model computes different numbers, so the tables it produces must never
stand in for the full model's. Each table records the run that produced it, and
`describe_table` reads it back: the quantized run carries its weight format, and
the two tables' definition hashes differ.

In [ ]:
dense_record, gguf_record = db.describe_table(table_dense), db.describe_table(table_gguf)
dense_run = dense_record["env"]["models"][0]["run"]
gguf_run = gguf_record["env"]["models"][0]["run"]
print(f"f32  run: quantization={dense_run.get('quantization')}   "
      f"definition {dense_record['definition_hash'][:16]}…")
print(f"q8_0 run: quantization={gguf_run.get('quantization')}   "
      f"definition {gguf_record['definition_hash'][:16]}…")

claim("the quantized run records its weight format and the dense run none",
      gguf_run.get("quantization") == "q8_0" and "quantization" not in dense_run)
claim("the two tables are two definitions",
      dense_record["definition_hash"] != gguf_record["definition_hash"])

## QLoRA: fine-tune over the quantized backbone

`fine_tune` takes a GGUF directory as its `base_model` too. With `target_modules`
naming the layers to adapt, it trains low-rank adapters beside the frozen quantized
weights: the gradient flows through the quantized matrices to the adapters, and the
quantized weights themselves are never updated. The base artifact chooses this path;
there is no separate flag.

The training pairs are 30 scored text pairs; a fifth is held out, and the held-out
loss is measured after every epoch.

In [ ]:
db.add_source("pairs", url=fixtures.url("tiny_pairs.csv"), format="csv")
job = db.fine_tune(
    source="pairs",
    base_model=f"local:{gguf_dir}",
    columns=["text_a", "text_b", "score"],
    method="lora",
    task="text_embedding",
    target_modules=["query", "value"],
    lora_rank=8,
    epochs=8,
    warmup_steps=0,
    learning_rate=1e-3,
    validation_fraction=0.2,
    early_stopping_metric="val_loss",
    seed=0,
)
job.wait()
metrics = job.metrics()
held_out = [round(row["loss"], 4) for row in metrics["val_loss_curve"]]
print(f"held-out loss per epoch: {held_out}")
print(f"the model kept: held-out loss {metrics['final_loss']:.4f}")

vector = db.encode_query(model=job.output_model_id, query="quantum computing applications")
claim("the adapted model encodes in the base model's width", len(vector) == 384)
claim("training over the quantized backbone lowers the held-out loss",
      min(held_out[1:]) < held_out[0], f"first {held_out[0]}, best after {min(held_out[1:])}")
claim("the model kept is the epoch with the lowest held-out loss",
      abs(metrics["final_loss"] - min(held_out)) < 1e-3)

Thirty pairs are few: the held-out loss falls, then turns up again as the adapters
begin to memorize the training pairs. Early stopping watches the held-out loss and
restores the best epoch's adapters, so the model a job publishes is the one that
generalized best, not the last one trained.

In [ ]:
db.close()

## References

- Dettmers, Tim, Pagnoni, Artidoro, Holtzman, Ari, Zettlemoyer, Luke (2023) *QLoRA: Efficient Finetuning of Quantized LLMs* Advances in Neural Information Processing Systems (NeurIPS) arXiv:2305.14314.